# Flow Chart Pengerjaan:
1. Cover Image
2. DCT & Quantization
3. Block Smoothness Estimation & Sorting
4. Zigzag Scan
5. NACP Construction
6. Adaptive Hexagonal Payload Assignment
7. Hexagonal Turtle Shell Embedding
8. Stego DCT Coefficients
9. Entropy Coding
10. Stego Image

In [125]:
!pip install jpeglib numpy matplotlib opencv-python-headless scipy scikit-image seaborn pandas tqdm import-ipynb


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [126]:
from PIL import Image
from performance import psnr, fsi, ssim
from zigzag import zigzag, inverse_zigzag
from math import ceil, floor, log2, log10, sqrt
import cv2
import jpeglib
import numpy as np
import import_ipynb
import matplotlib.pyplot as plt
import turtleShell
import FrequencyDomain as FD

In [ ]:
def change_image_QF(image_path, target_qf):
    im = jpeglib.read_dct(image_path)
    old_qt = im.qt[0]

    # From QF 100 to target QF
    dequantized = im.Y.astype(np.float64) * old_qt
    new_coefficients = np.round(dequantized / FD.custom_q_mat(target_qf)).astype(np.int16)
    
    # Update image object
    im.Y[:] = new_coefficients
    im.qt[0] = FD.custom_q_mat(target_qf)
    
    ori_path =  image_path.split(".jpeg")[0]
    output_path = f"{ori_path}_qf{target_qf}.jpeg"
    im.write_dct(output_path)

In [128]:
def get_compress_coeff(image_path, target_qf):
    im = jpeglib.read_dct(image_path)
    old_qt = im.qt[0]
    dequantized = im.Y.astype(np.float64) * old_qt
    new_coefficients = np.round(dequantized / FD.custom_q_mat(target_qf)).astype(np.int16)
    im.Y[:] = new_coefficients
    num_v_blocks, num_h_blocks, _, _ = im.Y.shape
    sorted_coeffs = []
    for i in range(num_v_blocks):
        for j in range(num_h_blocks):
            block = im.Y[i, j]
            zigzag_coeffs = zigzag(block)
            sorted_coeffs.append(zigzag_coeffs)
    return sorted_coeffs

In [129]:
def convert_tiff_to_jpeg(tiff_path, jpeg_path, quality=100):
    with Image.open(tiff_path) as img:
        rgb_img = img.convert('L')
        rgb_img.save(jpeg_path, 'JPEG', quality=quality)
    print(f"Converted {tiff_path} to {jpeg_path} with quality {quality}")

In [130]:
def sort_smoothness(smoothness_list):
    smoothness_list.sort(key=lambda x: (-x[1], x[2]))
    return smoothness_list

In [131]:
def block_smoothness_estimation(image):
    im = jpeglib.read_dct(image)
    h, w, _, _ = im.Y.shape
    smoothness_block = []
    total_ec = 0
    for i in range(h):
        for j in range(w):
            block = im.Y[i, j]
            block_1d = block.flatten()
            block_1d = block_1d[1:]  # AC coefficients 
            zero_count = np.sum(block_1d == 0)
            non_zero_sum = np.sum(abs(block_1d[block_1d != 0]))
            non_zero_indices = np.nonzero(block_1d)[0]
            capable_bits = 4 if zero_count == 0 else 3
            total_ec += len(non_zero_indices) // 2 * capable_bits
            smoothness_block.append(((i, j), zero_count, non_zero_sum))

    print(f"Total blocks: {h * w}")
    print(f"Total embedding capacity (estimated): {total_ec} bits")
    return smoothness_block

def block_smoothness(image):
    im = jpeglib.read_dct(image)
    h, w, _, _ = im.Y.shape
    q_table = im.qt[0]
    smoothness_block = []
    smoothness_score = []
    for i in range(h):
        for j in range(w):
            block = im.Y[i, j]
            ac_block = block.copy()
            ac_block[0, 0] = 0
            z_k = np.sum(ac_block == 0)
            E_k = np.sum((ac_block != 0) * (q_table ** 2))
            S_k = z_k + float(z_k / E_k)
            smoothness_block.append(((i, j), z_k, E_k, S_k))
            smoothness_score.append(((i, j), S_k))
    return smoothness_block, smoothness_score

In [132]:
def get_nacp(sorted_coefficients):
    valid_nacp = []
    for zigzag_coeff in sorted_coefficients:
        ac_coeffs = zigzag_coeff[1:] # AC coefficients
        non_zero_indices = np.nonzero(np.abs(ac_coeffs) >= 1)[0]
        non_zero_ac = [ac_coeffs[i] for i in non_zero_indices]

        for i in range(0, len(non_zero_ac) - 1, 2):
            x = int(non_zero_ac[i])
            y = int(non_zero_ac[i+1])
            if x != 0 and y != 0:
                valid_nacp.append((x, y))

    return valid_nacp

In [133]:
def replace_nacp(sorted_coefficients, nacp_coords):
    pair_index = 0

    for zigzag_coeff in sorted_coefficients:
        ac_coeffs = zigzag_coeff[1:]  
        non_zero_indices = np.nonzero(np.abs(ac_coeffs) >= 1)[0]
        non_zero_ac = [ac_coeffs[i] for i in non_zero_indices]

        for idx in range(0, len(non_zero_ac) - 1, 2):
            if pair_index < len(nacp_coords):
                new_x, new_y = nacp_coords[pair_index]
                i1, i2 = non_zero_indices[idx], non_zero_indices[idx + 1]
                ac_coeffs[i1] = float(new_x)
                ac_coeffs[i2] = float(new_y)
                pair_index += 1
            else: break
        zigzag_coeff[1:] = ac_coeffs

    return sorted_coefficients

In [134]:
def get_quantized_coefficients(image_path):
    im = jpeglib.read_dct(image_path)
    num_v_blocks, num_h_blocks, _, _  = im.Y.shape
    sorted_coeffs = []
    for i in range(num_v_blocks):
        for j in range(num_h_blocks):
            block = im.Y[i, j]
            zigzag_coeffs = zigzag(block)
            sorted_coeffs.append(zigzag_coeffs)
    return sorted_coeffs

In [135]:
def construct_stego_file(image_path, new_coeffs, qf=None):
    im = jpeglib.read_dct(image_path)
    num_v_blocks, num_h_blocks, v_block_size, h_block_size  = im.Y.shape
    idx = 0
    
    for i in range(num_v_blocks):
        for j in range(num_h_blocks):
            block_coeffs = new_coeffs[idx]
            block = inverse_zigzag(block_coeffs, v_block_size, h_block_size)
            im.Y[i, j] = block
            idx += 1

    if qf is not None:    
        dequantized = im.Y.astype(np.float64) * qf
        im.Y[:] = np.round(dequantized / FD.custom_q_mat(100)).astype(np.int16)
        im.qt[0] = FD.custom_q_mat(100)

    output_path = "stego-images/stego_" + image_path.split("/")[-1]
    print(f"Image with secret data is saved to {output_path}")
    im.write_dct(output_path)

In [136]:
def data_hiding_process(secret_data, nacp_coord="", mode="8N"):
    bit = 3 if mode == "8N" else 4
    secret_data = secret_data + '\0'
    data_bin = ''.join(format(ord(c), '08b') for c in secret_data)
    lendata = len(data_bin)
    print(f"Secret Data: {secret_data}")
    print(f"Panjang Bit Secret Data: {lendata}")

    decimals = []
    for i in range(0, len(data_bin), bit):
        group = data_bin[i:i+bit].ljust(bit, '0')
        decimals.append(int(group, 2))

    _, shells, cell_to_shells = turtleShell.init(mode) 
    if len(decimals) > len(nacp_coord):
        print("Warning: Not enough NACP coordinates to embed all data.")
        
    for i in range(len(decimals)):
        x, y = nacp_coord[i]
        if turtleShell.get_hex_matrix_value(x, y, mode) == decimals[i]:
            nacp_coord[i] = (x, y)
        else:
            # shell_coords = turtleShell.get_kxk_nearest_signed(x, y, 3)
            _, shell_coords = turtleShell.get_shell_coords(x, y, shells, cell_to_shells)
            found = turtleShell.find_corresponding_val(shell_coords, decimals[i], nacp_coord[i], mode)
            nacp_coord[i] = found
            
    return nacp_coord

In [137]:
def data_extract_process(nacp_coord, mode="8N"):
    extracted_data = ""
    data_bits = ""

    for i, (x, y) in enumerate(nacp_coord):
        val = turtleShell.get_hex_matrix_value(x, y, mode=mode)
        bit = 3 if mode == "8N" else 4
        bits = format(val & ((1 << bit) - 1), f'0{bit}b')
        data_bits += bits

        while len(data_bits) >= 8:
            byte = data_bits[:8]
            char_val = int(byte, 2)
            if char_val == 0: # Null terminator ASCII
                return extracted_data
            try:
                char = chr(char_val)
                extracted_data += char
            except:
                return extracted_data
            data_bits = data_bits[8:]

    return extracted_data

In [ ]:
def encode(image_path, message_bits):
    sorted_coeffs = get_quantized_coefficients(image_path)
    nacp_coords = get_nacp(sorted_coeffs)
    print(f"NACP Length: {len(nacp_coords)}")
    print(f"Total EC: {len(nacp_coords * 3)}")

    modified_nacp_coords = data_hiding_process(message_bits, nacp_coords, mode="8N")
    modified_coeffs = replace_nacp(sorted_coeffs, modified_nacp_coords)
    construct_stego_file(image_path, modified_coeffs)
    print("Data embedding completed.")

def encode_2(image_path, data, qf):
    image = Image.open(image_path).convert('L')
    stegoimg = image.copy()
    img_arr = np.array(stegoimg)
    q_mat = FD.custom_q_mat(qf)
    sorted_coefficients = FD.transform_to_freq(img_arr, q_mat)
    nacp_coords = get_nacp(sorted_coefficients)
    modified_nacp_coords = data_hiding_process(data, nacp_coords)
    modified_coeffs = replace_nacp(sorted_coefficients, modified_nacp_coords)
    np.save("modified_coefficients.npy", modified_coeffs)
    
def encode_3(image_path, secret_data):
    _, smoothness_score = block_smoothness(image_path)
    smoothness_list = sorted(smoothness_score, key=lambda x: -x[1])
    secret_data += '\0'
    data_bin = ''.join(format(ord(c), '08b') for c in secret_data)
    lendata = len(data_bin)

    _, shells, cell_to_shells = turtleShell.init(mode="8N")
    _, shells_17, cell_to_shells_17 = turtleShell.init(mode="17N")
    
    im = jpeglib.read_dct(image_path)
    for (block_i, block_j), score in smoothness_list:
        # print(f"Processing block ({block_i}, {block_j}) with smoothness score {score}")
        if lendata <= 0: break
        N = 8 if score <= 50 else 17
        t = 3 if N == 8 else 4
        mode = f"{N}N"
        block = im.Y[block_i, block_j]
        zigzag_coeffs = zigzag(block)
        ac_coeffs = zigzag_coeffs[1:]
        non_zero_indices = np.nonzero(np.abs(ac_coeffs) >= 1)[0]
        choosen_shells = shells if N == 8 else shells_17
        choosen_cell_to_shells = cell_to_shells if N == 8 else cell_to_shells_17
        for idx in range(0, len(non_zero_indices) - 1, 2):
            if lendata <= 0: break
            x = int(ac_coeffs[non_zero_indices[idx]])
            y = int(ac_coeffs[non_zero_indices[idx + 1]])
            # if idx >= 2:
            bits = data_bin[:t].ljust(t, '0') 
            data_bin = data_bin[t:]
            lendata -= t
            target_val = int(bits, 2)
            # else: target_val = t
            val_int = turtleShell.get_hex_matrix_value(x, y, mode=mode)
            if target_val != val_int:
                _, shell_coords = turtleShell.get_shell_coords(x, y, choosen_shells, choosen_cell_to_shells)
                x, y = turtleShell.find_corresponding_val(shell_coords, target_val, (x, y), mode=mode)
            ac_coeffs[non_zero_indices[idx]] = float(x)
            ac_coeffs[non_zero_indices[idx + 1]] = float(y)

        zigzag_coeffs[1:] = ac_coeffs
        zigzag_coeffs[0] = block[0, 0]  
        im.Y[block_i, block_j] = inverse_zigzag(zigzag_coeffs, 8, 8)

    output_path = "stego-images/stego_" + image_path.split("/")[-1]
    print(f"Data embedding completed. Stego image saved to {output_path}")
    im.write_dct(output_path)

In [139]:
def decode(stego_image_path):
    sorted_coeffs = get_quantized_coefficients(stego_image_path)
    nacp_coords = get_nacp(sorted_coeffs)
    extracted_bits = data_extract_process(nacp_coords, mode="8N")
    return extracted_bits

def decode_2(stego_file):
    modified_coeffs = np.load(stego_file, allow_pickle=True)
    nacp_coords = get_nacp(modified_coeffs)
    extracted_data = data_extract_process(nacp_coords)
    return extracted_data

def decode_3(stego_image_path):
    _, smoothness_score = block_smoothness(stego_image_path)
    smoothness_list = sorted(smoothness_score, key=lambda x: -x[1])
    im = jpeglib.read_dct(stego_image_path)
    bitstream = ""
    decoded_text = ""
    for (block_i, block_j), score in smoothness_list:
        N = 8 if score <= 50 else 17
        t = 3 if N == 8 else 4
        mode = f"{N}N"
        block = im.Y[block_i, block_j]
        zigzag_coeffs = zigzag(block)
        ac_coeffs = zigzag_coeffs[1:]
        non_zero_indices = np.nonzero(np.abs(ac_coeffs) >= 1)[0]

        for idx in range(0, len(non_zero_indices) - 1, 2):
            x = int(ac_coeffs[non_zero_indices[idx]])
            y = int(ac_coeffs[non_zero_indices[idx + 1]])
            val = turtleShell.get_hex_matrix_value(x, y, mode=mode)
            bits = format(val, f"0{t}b")
            bitstream += bits
            while len(bitstream) >= 8:
                byte = bitstream[:8]
                bitstream = bitstream[8:]
                char_val = int(byte, 2)
                if char_val == 0:   # Null terminator
                    return decoded_text
                decoded_text += chr(char_val)
    return decoded_text

In [140]:
def read_text_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        content = file.read()
    return content

In [ ]:
# convert_tiff_to_jpeg("cover-images/gray_boat.tiff", "cover-images/gray_boat.jpeg", quality=100)

Converted cover-images/gray_boat.tiff to cover-images/gray_boat.jpeg with quality 100


In [ ]:
pay_size = 100
cover_folder = "cover-images/"
stego_folder = "stego-images/"
payload_folder = "payload/"
cover_image_path = f"gray_baboon_qf70.jpeg"
stego_image_path = f"stego_gray_baboon_qf70.jpeg"
data = read_text_file(f"{payload_folder}{pay_size}Kb.txt")
encode_3(f"{cover_folder}{cover_image_path}", data)

secret_data = decode_3(f"{stego_folder}{stego_image_path}")
print("Extracted Data:", secret_data)

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_26540\505574587.py:35: RuntimeWarning: divide by zero encountered in divide
  S_k = z_k + float(z_k / E_k)


Processing block (22, 33) with smoothness score inf
Processing block (30, 33) with smoothness score 63.984375
Processing block (52, 25) with smoothness score 61.376543209876544
Processing block (31, 27) with smoothness score 61.286384976525824
Processing block (16, 39) with smoothness score 61.22592592592593
Processing block (53, 31) with smoothness score 60.32786885245902
Processing block (27, 28) with smoothness score 60.303030303030305
Processing block (50, 39) with smoothness score 60.303030303030305
Processing block (23, 33) with smoothness score 60.28436018957346
Processing block (31, 28) with smoothness score 60.28169014084507
Processing block (47, 20) with smoothness score 60.256410256410255
Processing block (27, 37) with smoothness score 60.24291497975708
Processing block (32, 27) with smoothness score 60.24096385542169
Processing block (30, 39) with smoothness score 60.229007633587784
Processing block (17, 38) with smoothness score 60.15706806282723
Processing block (53, 33) 

In [143]:
# Test performance metrics
psnr_value = psnr(f"{cover_folder}{cover_image_path}", f"{stego_folder}{stego_image_path}")
fsi_value = fsi(f"{cover_folder}{cover_image_path}", f"{stego_folder}{stego_image_path}")
ssim_value = ssim(f"{cover_folder}{cover_image_path}", f"{stego_folder}{stego_image_path}")
print(f"PSNR: {psnr_value} dB")
print(f"FSI: {fsi_value}")
print(f"SSIM: {ssim_value}")

Size cover: 62332
Size stego: 65508
PSNR: 26.853966715179038 dB
FSI: 0.050952961560675095
SSIM: 0.825141788064033


In [144]:
def compare_spatial_frequency(cover_image_path, stego_image_path):
    cover = np.array(Image.open(cover_image_path).convert('L'), dtype=np.float64)
    stego = np.array(Image.open(stego_image_path).convert('L'), dtype=np.float64)

    spatial_difference = np.abs(cover - stego) ** 2

    cover_freq = np.fft.fft2(cover)
    stego_freq = np.fft.fft2(stego)
    freq_difference = np.abs(cover_freq - stego_freq)

    return freq_difference, spatial_difference

In [ ]:
# freq_diff, spatial_diff = compare_spatial_frequency(f"{cover_folder}{cover_image_path}", f"{stego_folder}{stego_image_path}")
# print("Frequency Difference:")
# for row in freq_diff:
#     print(row)
# print("Spatial Difference:")
# for row in spatial_diff:
#     print(row)
# print("Max spatial diff:", spatial_diff.max())
# print("Mean spatial diff:", spatial_diff.mean())

Frequency Difference:
[ 1410.           856.04330321   468.14318737   324.2815383
   337.49795738   312.06203753   122.97834958   680.70712862
   789.67473058  1416.44777012   964.18977256   335.80261896
  1991.18166153  2541.41584638   651.62476252  2332.67345116
  1582.99044997   753.99530204  1098.95035626   947.18695048
  1184.1172753   3091.1645331    824.61956128  2318.47749795
  1782.52564798  4080.92593272  2440.032911     823.53600492
  4795.84955294   460.07662395  1686.71013104  1225.26783195
  2335.71365597  2221.81063283   724.84743974  1586.4830054
  5294.20985427  3923.00635831  5718.30513221  5418.34197005
  4760.50545313  1110.18418973  1699.66765879  3165.76063281
  3570.45728863  1272.63324623  2944.67191425  3100.2693056
  2961.70987398  3891.50366244  1834.9137424   4521.99157256
   988.16304721  1619.90154465  3883.64254604  1504.15361846
  2229.72283411  3885.30816863  1621.37857541  5541.00576699
  4597.22509689  2660.4909078   3778.87633447  4086.5471597
  2508

In [146]:
max_pixel = 255.0
psnr_value = 20 * log10(max_pixel / sqrt(spatial_diff.mean()))
print("PSNR calculated from spatial difference:", psnr_value, "dB")

PSNR calculated from spatial difference: 26.853966715179038 dB
